# FinGPT-India: Month 2 — QLoRA fine-tuning (baseline FinLLM)

Run `datacollection.ipynb` first so the `chunks` list exists in the same session, or load chunks from a saved file.

Stand-in model: Qwen2.5-3B-Instruct (LLaMA 3.1 8B is too large for free Colab GPU). Say so in the report.

## Step 1: Generate SFT data (GPU)

In [ ]:
import json, random, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
gen_model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="auto")

random.seed(42)
sample = random.sample(chunks, 300)

PROMPT = """Read the passage and write ONE question a retail investor might ask that the passage answers, then the answer.
Use only facts in the passage. Reply exactly in this format:
Q: <question>
A: <answer>

Passage:
{passage}"""

def make_qa(passage):
    msgs = [{"role": "user", "content": PROMPT.format(passage=passage[:3000])}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(gen_model.device)
    out = gen_model.generate(ids, max_new_tokens=200, do_sample=False)
    text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    if "Q:" in text and "A:" in text:
        q = text.split("Q:", 1)[1].split("A:", 1)[0].strip()
        a = text.split("A:", 1)[1].strip()
        return q, a
    return None, None

rows, failed = [], 0
for c in sample:
    q, a = make_qa(c["text"])
    if not q or not a or len(a) <= 20:
        failed += 1
        continue
    rows.append({"messages": [{"role": "user", "content": q},
                              {"role": "assistant", "content": a}],
                 "source": c["doc"]})

with open("sft.jsonl", "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"kept {len(rows)} of 300; discarded {failed} (format or length)")
del gen_model; torch.cuda.empty_cache()


## Step 2: Check 10 random examples by hand

Read each answer against its source. Record how many are not supported; these count as discards in the report.

In [ ]:
import random
lines = [json.loads(l) for l in open("sft.jsonl", encoding="utf-8")]
random.seed(7)
for r in random.sample(lines, min(10, len(lines))):
    print("SOURCE:", r["source"])
    print("Q:", r["messages"][0]["content"])
    print("A:", r["messages"][1]["content"][:300])
    print("-" * 60)


## Step 3: QLoRA fine-tuning (GPU)

Mount Drive so the adapter survives a disconnect. Log the example count, epochs, learning rate, GPU and final loss for the report.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

BASE = "Qwen/Qwen2.5-3B-Instruct"
OUT = "/content/drive/MyDrive/fingpt-india/qlora-qwen2.5-3b"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")

ds = load_dataset("json", data_files="sft.jsonl", split="train").remove_columns(["source"])

peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])

args = SFTConfig(output_dir=OUT, num_train_epochs=2, per_device_train_batch_size=2,
                 gradient_accumulation_steps=4, learning_rate=2e-4, lr_scheduler_type="cosine",
                 warmup_ratio=0.03, logging_steps=5, save_strategy="epoch",
                 fp16=True, report_to="none")

trainer = SFTTrainer(model=model, args=args, train_dataset=ds,
                     processing_class=tok, peft_config=peft_cfg)
result = trainer.train()
trainer.save_model(OUT)
print("training loss:", result.training_loss, "| examples:", len(ds), "| epochs: 2")
